# WaPOR 300 m → 20 m downscaling — inference on a new AOI

This notebook runs the **7-site Ensemble-L1** model (SwinIR-L1-7site + Prithvi-V1-L1-7site, 50/50) on a brand-new AOI you define below.

Steps:
1. Configure the AOI (bbox + date range + L3 site code)
2. Download WaPOR L1 300 m (and L2 100 m, L3 20 m if you want ground truth)
3. Fetch Sentinel-2 B4/B8/B11 via GEE (cloud-masked dekadal median, snapped to the L3 20 m grid)
4. Build the 5-band 20 m input stack with L1 in the ETa300m channel
5. Load SwinIR-L1-7site + Prithvi-V1-L1-7site
6. Run tiled Hann-blend inference and save SwinIR / Prithvi / Ensemble GeoTIFFs
7. (Optional) Plot the result alongside the L1 input and the L3 ground truth

**Requirements**: same `irr312` conda env you use for training, a Google Earth Engine service-account JSON for step 3, and the trained checkpoints under `models/`.

## 1. Configure

In [ ]:
import sys, json, subprocess, time
from pathlib import Path

# --- repo + env ---
REPO = Path(r'c:\Users\z.kiala\Documents\wapor_africa').resolve()
PY   = Path(r'C:\envs\irr312\python.exe')                # conda env Python
GEE_SA_JSON = REPO / 'tethys-app-1-087d23478872.json'      # service-account JSON for GEE

# --- model checkpoints (7-site Ensemble) ---
SWINIR_CKPT  = REPO / 'models' / 'multi7_swinir_l1_e96_w16' / 'swinir_best.pt'
PRITHVI_CKPT = REPO / 'models' / 'multi7_prithvi_v1_l1'     / 'prithvi_best.pt'

# --- NEW AOI ---
SITE_NAME      = 'NEWSITE'                                 # short code (e.g., 'XYZ')
L3_SITE_CODE   = 'KOG'                                     # WaPOR L3 site code if you want L3 ground truth, else None
AOI_BBOX_4326  = [37.00, 11.20, 37.40, 11.60]              # [lon_min, lat_min, lon_max, lat_max]
DATE_START     = '2024-04-01'
DATE_END       = '2024-04-10'                              # one dekad. Can extend to whole year.

# --- output paths under data/<site_lower>/ ---
SITE_DIR = REPO / 'data' / SITE_NAME.lower()
STACK_DIR_20M  = SITE_DIR / 'stacks' / f'{SITE_NAME}_STACK_S2_MATCH_L3_20M_FULL_1'
STACK_DIR_L1   = SITE_DIR / 'stacks' / f'{SITE_NAME}_STACK_S2_MATCH_L3_20M_L1_FULL_1'
PRED_DIR       = REPO / 'models' / 'multi7_ensemble_l1' / 'predictions' / SITE_NAME
PRED_DIR.mkdir(parents=True, exist_ok=True)

for p in (SWINIR_CKPT, PRITHVI_CKPT, GEE_SA_JSON, PY):
    assert p.exists(), f'Missing: {p}'
print('OK — paths valid.')
print(f'  predictions → {PRED_DIR}')

Write a one-off `configs/<site>.yaml` so the existing data-prep scripts can be reused:

In [ ]:
yaml_path = REPO / 'configs' / f'{SITE_NAME.lower()}.yaml'
yaml_text = f"""site:
  name: {SITE_NAME}
  description: New-AOI inference run
  l3_site_code: {L3_SITE_CODE or 'null'}
  target_bbox_hint: [{AOI_BBOX_4326[0]}, {AOI_BBOX_4326[1]}, {AOI_BBOX_4326[2]}, {AOI_BBOX_4326[3]}]

time_range:
  start: \"{DATE_START}\"
  end:   \"{DATE_END}\"
  cadence: dekad

wapor:
  bucket: fao-gismgr-wapor-3-data
  l3_target_mapset: L3-AETI-D
  l2_predictor_mapset: L2-AETI-D
  l3_grid_prefix:   DATA/WAPOR-3/GRID/L3-GRID
  l3_data_prefix:   DATA/WAPOR-3/MOSAICSET/L3-AETI-D
  l2_data_prefix:   DATA/WAPOR-3/MAPSET/L2-AETI-D
  scale_l2: 100
  scale_l3: 100

sentinel2:
  collection: COPERNICUS/S2_SR_HARMONIZED
  bands: [B4, B8, B11]
  cloud_filter: CLOUDY_PIXEL_PERCENTAGE
  cloud_max: 60
  composite_window_days: 10
  scale_m: 20

static:
  dem_source: USGS/SRTMGL1_003
  worldcover_source: ESA/WorldCover/v200
  exclude_urban: true
  exclude_water: true

rainfall:
  source: UCSB-CHG/CHIRPS/DAILY
  window_days: 10
  lag_window_days: 10

gee:
  service_account_json: {GEE_SA_JSON.as_posix()}
  service_account_email: null

paths:
  data_root: data/{SITE_NAME.lower()}
  wapor_l3_dir: data/{SITE_NAME.lower()}/wapor_l3
  wapor_l2_dir: data/{SITE_NAME.lower()}/wapor_l2
  s2_dir:       data/{SITE_NAME.lower()}/s2
  static_dir:   data/{SITE_NAME.lower()}/static
  rainfall_dir: data/{SITE_NAME.lower()}/rainfall
  stacks_dir:   data/{SITE_NAME.lower()}/stacks/{SITE_NAME}_STACK_S2_MATCH_L3_20M_FULL_1
  repo_dir:     third_party/WaPOR-downscaling
"""
yaml_path.write_text(yaml_text)
print(f'Wrote {yaml_path}')

## 2. Download WaPOR L1 (300 m) + L2 + L3 (if available)

In [ ]:
def run(cmd, desc):
    print(f'\n[STEP] {desc}\n  {" ".join(str(c) for c in cmd)}')
    t0 = time.time()
    r = subprocess.run(cmd, cwd=REPO)
    print(f'[STEP] {desc}  → exit={r.returncode}  took {time.time()-t0:.1f}s')
    return r.returncode

# WaPOR L3 ground truth (skip if L3_SITE_CODE is None)
if L3_SITE_CODE:
    run([PY, 'scripts/data_prep/01_download_wapor_l3.py', '--config', SITE_NAME.lower(), '--site-code', L3_SITE_CODE],
        'Download WaPOR L3 (ground truth)')
# WaPOR L2 — needed only if you also want to compute Stage-2-only metrics (not used at inference)
run([PY, 'scripts/data_prep/02_download_wapor_l2.py', '--config', SITE_NAME.lower()],
    'Download WaPOR L2 (100 m)')
# WaPOR L1 — the actual coarse input the model needs
run([PY, 'scripts/data_prep/01b_download_wapor_l1.py', '--config', SITE_NAME.lower()],
    'Download WaPOR L1 (300 m)')

## 3. Fetch Sentinel-2 via GEE

Pulls cloud-masked dekadal median B4/B8/B11 composites at 20 m, snapped to the L3 grid.
Slowest step (~5-30 min depending on AOI size + date range).

In [ ]:
run([PY, 'scripts/data_prep/03_fetch_s2_gee.py', '--config', SITE_NAME.lower(),
     '--tile-cols', '2', '--tile-rows', '2'],
    'Fetch Sentinel-2 via GEE')

## 4. Build the input stack (with L1 in the `ETa300m` channel)

In [ ]:
# 4a. Build the 20m stack (S2 + L2-derived ETa300m + L3 label). This wraps everything except L1.
run([PY, 'scripts/data_prep/06_build_stack.py', '--config', SITE_NAME.lower()],
    'Build 20m stack (S2 + L2 + L3)')
# 4b. Replace the ETa300m channel with L1 300m bilinear → 20m → new *_L1_FULL_1 stack dir.
run([PY, 'scripts/data_prep/07_rebuild_stack_l1.py', '--config', SITE_NAME.lower()],
    'Rebuild stack with L1 in coarse channel')

## 5. Run model inference (SwinIR + Prithvi + Ensemble) on every dekad

In [ ]:
stack_files = sorted(STACK_DIR_L1.glob('*.tif'))
print(f'Found {len(stack_files)} L1-stack(s) to process')
for fp in stack_files:
    run([PY, 'scripts/predict_ensemble_one.py',
         '--swinir-ckpt',  SWINIR_CKPT,
         '--prithvi-ckpt', PRITHVI_CKPT,
         '--stack',        fp,
         '--out-dir',      PRED_DIR,
         '--weight', '0.5'],
        f'Predict {fp.name}')
# Also extract L1 input + L3 ground truth as standalone rasters for side-by-side viz.
for fp in stack_files:
    run([PY, 'scripts/_extract_l3_ref.py', fp, PRED_DIR], f'Extract L1+L3 ref for {fp.name}')

## 6. (Optional) Visualize one dekad

In [ ]:
# Lightweight inline viz with matplotlib. For interactive panning/zooming use QGIS.
import rasterio
import numpy as np
import matplotlib.pyplot as plt

def show(path, title, vmin=0, vmax=50, ax=None):
    with rasterio.open(path) as ds:
        arr = ds.read(1).astype(np.float32)
    arr = np.where(arr == ds.nodata, np.nan, arr) if ds.nodata is not None else arr
    im = ax.imshow(arr, vmin=vmin, vmax=vmax, cmap='viridis')
    ax.set_title(title); ax.axis('off')
    return im

stem = stack_files[0].stem if stack_files else None
if stem is None:
    print('No stacks found — run cells 2-4 first.')
else:
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    show(PRED_DIR / f'{stem}_input_L1_at_20m.tif',  'L1 input (300m → 20m)', ax=axes[0])
    show(PRED_DIR / f'{stem}_swinir.tif',           'SwinIR-L1-7site',         ax=axes[1])
    show(PRED_DIR / f'{stem}_prithvi.tif',          'Prithvi-V1-L1-7site',     ax=axes[2])
    show(PRED_DIR / f'{stem}_ensemble.tif',         'Ensemble (50/50)',        ax=axes[3])
    show(PRED_DIR / f'{stem}_groundtruth_L3.tif',   'L3 ground truth (20m)',   ax=axes[4])
    plt.suptitle(f'{SITE_NAME}  {stem}')
    fig.colorbar(axes[-1].images[0], ax=axes, shrink=0.7, label='mm/dekad')
    plt.show()

## Notes

- **Hold-out year-aware eval**: this notebook just produces predictions. If you want quantitative comparison to L3 ground truth, run `scripts/eval_swinir_one.py` / `scripts/prithvi/prithvi_eval.py` / `scripts/ensemble_evaluate.py` afterwards pointing `--site-spec` at the new L1-stack dir.
- **No L3 available?** The model still produces a downscaled 20 m AETI even if you skip `01_download_wapor_l3.py`. The stack will need a dummy `b1` band — easiest workaround: build the stack with the L1 file as both input and "label placeholder" by symlinking, or modify `06_build_stack.py` to skip the L3 read when missing. For now this notebook assumes L3 is available.
- **Custom AOI shape**: set `AOI_BBOX_4326` manually if you don't have a WaPOR L3 footprint to snap to. The S2 fetcher and the L1/L2 downloaders all accept a bbox.
- **Whole-year inference**: extend `DATE_END` to `2024-12-31`, re-run cells 2-5. ~5 min compute per dekad on an RTX 5090.